[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/89_grid_walk_flip_bd260723_solution.ipynb)

# 参考解法：格子行走翻转

Reference solution.

## 解析

**结论：把每个格子的状态拆成「素数底色」⊕「被翻转的奇偶」。逐个人模拟行走并翻转落脚格，最后统计奇偶为 1 的格子。**

### 状态表示
格子 `g` 的初始颜色只取决于它是否为素数；之后每次落脚会翻转一次。用 `parity[g]` 记录翻转次数的奇偶，当前颜色即 `is_prime(g) ^ parity[g]`，无需真正存储无限格子。

### 行走
第 `i` 个人消费 `s` 的前 `i` 个字符。每一步从当前格右侧起逐格找第一个颜色等于目标的格子（`'0'`→白、`'1'`→黑）。走完翻转落脚格的奇偶。颜色改变对后续的人可见，故必须按 1..n 顺序模拟。

### 验证
已用基于埃氏筛、直接存储颜色的独立实现在数百组随机 `(n, s)` 上对拍一致，并复现两个官方样例。

### 复杂度
设总步数与扫描距离有关；对题目给定的小规模数据，逐人模拟足以通过。素性判断用 `O(√g)` 试除。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List, Tuple

In [ ]:
# ✅ SOLUTION

def _is_prime(x: int) -> bool:
    if x < 2:
        return False
    if x < 4:
        return True
    if x % 2 == 0:
        return False
    i = 3
    while i * i <= x:
        if x % i == 0:
            return False
        i += 2
    return True

class Solution:
    def changed_grids(self, n: int, s: str) -> Tuple[int, List[int]]:
        parity = {}                                  # grid -> flip parity on top of prime base
        def col(g: int) -> int:
            return (1 if _is_prime(g) else 0) ^ parity.get(g, 0)
        for i in range(1, n + 1):
            cur = 1
            for c in s[:i]:
                need = 1 if c == '1' else 0          # '1' -> black, '0' -> white
                g = cur + 1
                while col(g) != need:
                    g += 1
                cur = g
            parity[cur] = parity.get(cur, 0) ^ 1   # flip the landing grid
        changed = sorted(g for g, p in parity.items() if p == 1)
        return len(changed), changed

In [ ]:
sol = Solution()
print(sol.changed_grids(1, "0"))    # (1, [4])
print(sol.changed_grids(2, "11"))   # (2, [2, 5])

In [ ]:
from torch_judge import check
check('grid_walk_flip_bd260723')